# **1. INTRODUCTION :**

In [ ]:
import sys

import itertools
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", None)

print("Interpréteur :", sys.executable)
print("Python       :", sys.version.split()[0])
print("pandas       :", pd.__version__)

In [ ]:
from datetime import date
from pathlib import Path

# ------------- PATH ----------------------------------------------------------------
PROJECT_PATH = "/marbec-data/MOOBYF_BIOD/WP5_Video-surveys/merge_and_split_annotation_pipeline"
RUN_DATE = f"{date.today():%Y%m%d}"
DATA_FOLDER_PATH = Path(f"{PROJECT_PATH}/data")
OLD_PATH = Path(f"{DATA_FOLDER_PATH}/old_annotations/crops_16cls.csv")
NEW_PATH = Path(f"{DATA_FOLDER_PATH}/new_annotations")
OUT_DIR = Path(
    f"{PROJECT_PATH}/output/{RUN_DATE}/"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------- CSV ----------------------------------------------------------------
SEP = ","
ENCODING = "utf-8"
DTYPE = "str"

In [ ]:
EXCLUDED_CLASS = ["other", "Thunnus_albacares", "Cantherhines_pardalis"]


CLASS = pd.DataFrame(
    {
        "class_id": range(22),
        "class_name": [
            "Abudefduf_vaigiensis",
            "Aluterus_monoceros",
            "Aluterus_scriptus",
            "Caranx_sexfasciatus",
            "Carcharhinus_falciformis",
            "Carcharhinus_longimanus",
            "Coryphaena_hippurus",
            "Decapterus_macarellus",
            "Diver",
            "Elagatis_bipinnulata",
            "Platax_teira",
            "Seriola_rivoliana",
            "Shark",
            "not_fish",
            "Cantherhines_sp",
            "Fish",
            "Canthidermis_maculatus",
            "Ferdauia_orthogrammus",
            "Kyphosus_vaigiensis",
            "Lobotes_surinamensis",
            "Sphyraena_barracuda",
            "Thunnus_sp",
        ],
    },
    dtype="str",
)

In [ ]:
old_annotations_df = pd.read_csv(OLD_PATH, sep=SEP, encoding=ENCODING, dtype=DTYPE)

old_annotations_df.head(5)


# **2. MERGE NEW LINES :**

Input : Splitted new csv files

Output : Merged new csv file : new_annotations_df

### **2.1. CODE :**

In [ ]:
new_annotations_df = pd.DataFrame()

for folder in NEW_PATH.iterdir():
    if folder.is_dir():
        for file in folder.iterdir():
            if file.suffix == ".csv":
                file_df = pd.read_csv(file, sep=SEP, encoding=ENCODING)
                file_df.insert(0, "source_folder", f"new_{folder.name}")
                file_df.insert(1, "source_file", f"new_{file.name}")
                file_df["is_corrected"] = np.where(
                    file_df["validation_source"] == "DINO", False, True
                )
                print("file:", file.name, file_df.shape)
                new_annotations_df = pd.concat(
                    [new_annotations_df, file_df], ignore_index=True
                )
                print(new_annotations_df.shape)

# get rid off not validated lines & duplicates

new_annotations_df = new_annotations_df[
    new_annotations_df["validated"] == "Yes"
].drop_duplicates(subset=["crop_filename", "final_label", "validated"])

new_annotations_df.to_csv(
    OUT_DIR / f"new_annotation_{RUN_DATE}.csv",
    sep=SEP,
    encoding=ENCODING,
    index=False,
)


In [ ]:
tab_detail = new_annotations_df.groupby(new_annotations_df["final_label"]).agg(
    n_total=("is_corrected", "count"),
    n_corrected=("is_corrected", "sum"),
)

tab_detail.loc["Total"] = tab_detail.sum()
tab_detail["perc_corrected"] = (
    tab_detail["n_corrected"] / tab_detail["n_total"] * 100
).round(1)


display(tab_detail)


In [ ]:
labels = sorted(
    set(new_annotations_df["final_label"].dropna())
    | set(new_annotations_df["DINO_Prediction"].dropna())
)

cm = pd.crosstab(
    new_annotations_df["final_label"],
    new_annotations_df["DINO_Prediction"],
).reindex(index=labels, columns=labels, fill_value=0)

cm_tot = cm.copy()
cm_tot.loc["Total"] = cm_tot.sum()
cm_tot["Total"] = cm_tot.sum(axis=1)

display(cm_tot)

### **2.2. FIGURES :**

In [ ]:
cm_pct = cm.div(cm.sum(axis=1), axis=0).mul(100).round(1)


fig, ax = plt.subplots(figsize=(0.6 * len(labels) + 3, 0.6 * len(labels) + 2))
im = ax.imshow(cm_pct.fillna(0).values, cmap="Blues", vmin=0, vmax=100)

ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
ax.set_yticks(range(len(labels)), labels)
ax.set_xlabel("DINO_Prediction")
ax.set_ylabel("final_label")

for i in range(len(labels)):
    for j in range(len(labels)):
        n = cm.iloc[i, j]
        if n:
            ax.text(
                j,
                i,
                n,
                ha="center",
                va="center",
                fontsize=8,
                color="white" if cm_pct.fillna(0).iloc[i, j] > 50 else "black",
            )

fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(0.5 * len(tab_detail) + 3, 4))
ax.bar(tab_detail.index.astype(str), tab_detail["perc_corrected"])

for x, (pct, n) in enumerate(zip(tab_detail["perc_corrected"], tab_detail["n_total"])):
    ax.text(x, pct + 1, f"n={n}", ha="center", fontsize=8)

ax.set_ylim(0, 110)
ax.set_ylabel("% of corrected annotations")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=45)
plt.setp(ax.get_xticklabels(), ha="right")
fig.tight_layout()
plt.show()


# **3. MERGE OLD AND NEW LINES :**

Input : Merged new csv file + Old csv file

Output : Merged final csv file


### **3.1. CODE :**

In [ ]:
new_annotation_df = pd.read_csv(
    OUT_DIR / f"new_annotation_{RUN_DATE}.csv",
    sep=SEP,
    encoding=ENCODING,
    dtype=DTYPE,
)
new_annotation_df.drop(
    columns=[
        "source_folder",
        "source_file",
        "source_stem",
        "YOLO_Prediction",
        "DINO_Prediction",
        "YOLO_Confidence",
        "DINO_Confidence",
        "validated",
        "validation_source",
        "is_corrected",
    ],
    inplace=True,
)

new_annotation_df["resolution"] = "1920x1080"

old_annotations_df = pd.read_csv(
    OLD_PATH, sep=SEP, encoding=ENCODING, dtype=DTYPE
).drop(columns="class_id")

final_annotations_df = pd.concat(
    [old_annotations_df, new_annotation_df], ignore_index=True
).rename(columns={"final_label": "class_name"})
print(final_annotations_df.shape)

final_annotations_df["class_name"] = final_annotations_df["class_name"].replace(
    to_replace=["fish"],
    value=[
        "Fish"
    ],  
)

coord_cols = ["x_min", "y_min", "x_max", "y_max"]
final_annotations_df[coord_cols] = final_annotations_df[coord_cols].apply(pd.to_numeric, errors="coerce")

wh = final_annotations_df["resolution"].str.extract(r"^(?P<W>\d+)x(?P<H>\d+)$").astype(float)

final_annotations_df["x_min_n"] = final_annotations_df["x_min"] / wh["W"]
final_annotations_df["x_max_n"] = final_annotations_df["x_max"] / wh["W"]
final_annotations_df["y_min_n"] = final_annotations_df["y_min"] / wh["H"]
final_annotations_df["y_max_n"] = final_annotations_df["y_max"] / wh["H"]

print(final_annotations_df.shape)
print(final_annotations_df.dtypes)


OUT_PATH_2 = OUT_DIR / f"new_and_old_annotation_{RUN_DATE}.csv"
final_annotations_df.to_csv(OUT_PATH_2, sep=SEP, encoding=ENCODING, index=False, float_format="%.6f")


### **3.2. FIGURES :**

In [ ]:
tab_final = final_annotations_df.groupby("class_name").agg(
    n_final=("class_name", "count")
).sort_values(by="n_final", ascending=False)

tab_final


In [ ]:
ax = tab_final["n_final"].plot(
    kind="bar", figsize=(10, 5), width=0.9, edgecolor="white"
)

ax.bar_label(ax.containers[0], padding=4, fontsize=9)

plt.xlabel("class")
plt.ylabel("instance")
plt.xticks(rotation=45, ha="right", rotation_mode="anchor")

plt.tight_layout()

plt.ylim(0, tab_final["n_final"].max() * 1.15)

plt.show()


# **4. SPLIT THE DATASET :**

Input : merged final csv file

Output : csv splitted file  80/20

In [ ]:
# Create the dataframe with the number of annotations per species per videos

final_annotations_filtered_df = final_annotations_df[
    ~final_annotations_df["class_name"].isin(EXCLUDED_CLASS)
]

final_annotations_filtered_df = final_annotations_filtered_df.merge(
    CLASS[["class_id", "class_name"]],
    on="class_name",
    how="left",
)

video_df = (
    final_annotations_filtered_df.groupby("source_video")
    .agg(n_annotations=("source_video", "count"))
    .sort_values("n_annotations", ascending=False)
)

labels_df = pd.crosstab(
    final_annotations_filtered_df["source_video"],
    final_annotations_filtered_df["class_name"],
)

video_df = video_df.join(labels_df)
display(video_df.head(5))

### **4.2 SPLIT ALGO :** 
Greedy algo : Scored on the distance from target nb of annotations at each step

In [ ]:
species_count = final_annotations_filtered_df.groupby("class_name").agg(
    n_total=("class_name", "count")
)

label_cols = [c for c in video_df.columns if c not in ("n_annotations", "subset")]
n_label = len(label_cols)

species_count["n_tra"] = video_df.iloc[0][label_cols]
species_count["n_val"] = 0

total = species_count["n_total"].copy()
target_tra = 0.8 * total
target_val = 0.2 * total

video_df["subset"] = ""
video_df.loc[video_df.index[0], "subset"] = "training"

for video in video_df.index[1:]:
    counts = video_df.loc[video, label_cols]

    n_tra_if = species_count["n_tra"] + counts
    n_val_if = species_count["n_val"] + counts

    cost_tra = (
        abs(n_tra_if - target_tra) / total
        + abs(species_count["n_val"] - target_val) / total
    ).sum()
    cost_val = (
        abs(species_count["n_tra"] - target_tra) / total
        + abs(n_val_if - target_val) / total
    ).sum()

    if cost_tra < cost_val:
        species_count["n_tra"] = n_tra_if
        video_df.loc[video, "subset"] = "training"
    else:
        species_count["n_val"] = n_val_if
        video_df.loc[video, "subset"] = "validation"

species_count["perc_tra"] = species_count["n_tra"] / species_count["n_total"]

print(species_count)
kpi = (species_count["perc_tra"] - 0.8).abs().mean() * 100
print(f"Average deviation at 80%: {kpi:.1f} points")

ann = video_df.groupby("subset")["n_annotations"].sum()
display((ann / ann.sum() * 100).round(1))

### **4.3 Arrangements :** 
if nb of validation annotation <10 => try new combination with concerned videos

In [ ]:
MIN = 10
counts = video_df[label_cols]
total = species_count["n_total"]

# Videos associated to species with nb_val < 10
low = species_count.index[species_count["n_val"] < MIN]
cand = counts.index[(counts[low] > 0).any(axis=1)]
print(f"{len(low)} species with nb_val < 10, {len(cand)} videos associated")

# Current status
fixed_val = counts.index[
    (video_df["subset"] == "validation") & ~counts.index.isin(cand)
]
base_val = counts.loc[fixed_val].sum()


def evaluate(combo):
    n_val = base_val + counts.loc[cand].mul(list(combo), axis=0).sum()
    n_tra = total - n_val
    return (
        (n_tra <= MIN).sum(),  # 1 :
        (n_val < MIN).sum(),  # 2 :
        (n_tra / total - 0.8).abs().mean() * 100,  # 3 :
    )


best_combo = min(itertools.product([0, 1], repeat=len(cand)), key=evaluate)

video_df.loc[cand, "subset"] = ["validation" if c else "training" for c in best_combo]


species_count["n_val"] = base_val + counts.loc[cand].mul(list(best_combo), axis=0).sum()
species_count["n_tra"] = species_count["n_total"] - species_count["n_val"]
species_count["perc_tra"] = species_count["n_tra"] / species_count["n_total"]

# PRINT
print(species_count)
kpi = (species_count["perc_tra"] - 0.8).abs().mean() * 100
print(f"Average deviation at 80%: {kpi:.1f} points")

ann = video_df.groupby("subset")["n_annotations"].sum()
display((ann / ann.sum() * 100).round(1))

In [ ]:
video_splitted_df = video_df[["subset", "n_annotations"]]
#print(video_splitted_df)

In [ ]:
final_annotations_filtered_df = final_annotations_filtered_df.drop(columns="subset", errors="ignore").join(
    video_splitted_df["subset"], on="source_video", how="left"
)

final_annotations_filtered_df["frame"] = (
    final_annotations_filtered_df["source_video"]
    + "_"
    + final_annotations_filtered_df["frame_idx"]
)

display(final_annotations_filtered_df.head(5))

OUT_PATH_3 = OUT_DIR / f"splitted_annotation_{RUN_DATE}.csv"
final_annotations_filtered_df.to_csv(
    OUT_PATH_3, sep=SEP, encoding=ENCODING, index=False
)

OUT_PATH_5 = OUT_DIR / f"splitted_frame_{RUN_DATE}.csv"
final_annotations_filtered_df[["frame", "subset"]].drop_duplicates().to_csv(
    OUT_PATH_5, sep=SEP, encoding=ENCODING, index=False
)


In [ ]:
print(final_annotations_filtered_df.shape)